<a href="https://colab.research.google.com/github/Talia-commits-creator/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

if not os.path.exists("/content/plp-da-a3-duka-sales-and-agent-cleanup"):
    !git clone https://github.com/Talia-commits-creator/plp-da-a3-duka-sales-and-agent-cleanup.git

os.chdir("/content/plp-da-a3-duka-sales-and-agent-cleanup")

!python generate_data.py

print("Project folder:", os.getcwd())
print("Agent data:", os.path.exists("agents.csv"))
print("Transaction data:", os.path.exists("agent_transactions.csv"))
print("Cleaning pipeline:", os.path.exists("cleaning_pipeline.py"))

Cloning into 'plp-da-a3-duka-sales-and-agent-cleanup'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 35.56 KiB | 5.93 MiB/s, done.
Resolving deltas: 100% (3/3), done.
python3: can't open file '/content/plp-da-a3-duka-sales-and-agent-cleanup/generate_data.py': [Errno 2] No such file or directory
Project folder: /content/plp-da-a3-duka-sales-and-agent-cleanup
Agent data: False
Transaction data: False
Cleaning pipeline: False


In [2]:
import os

for root, dirs, files in os.walk(
    "/content/plp-da-a3-duka-sales-and-agent-cleanup"
):
    level = root.replace(
        "/content/plp-da-a3-duka-sales-and-agent-cleanup", ""
    ).count(os.sep)

    print("  " * level + os.path.basename(root) + "/")

    for file in files:
        print("  " * (level + 1) + file)

plp-da-a3-duka-sales-and-agent-cleanup/
  duka_sales_exploration.ipynb
  README.md
  .git/
    packed-refs
    HEAD
    description
    index
    config
    objects/
      info/
      pack/
        pack-9f316aa35696729382fb6426cfe5e0a025ed8719.pack
        pack-9f316aa35696729382fb6426cfe5e0a025ed8719.idx
        pack-9f316aa35696729382fb6426cfe5e0a025ed8719.rev
    logs/
      HEAD
      refs/
        heads/
          main
        remotes/
          origin/
            HEAD
    refs/
      heads/
        main
      tags/
      remotes/
        origin/
          HEAD
    hooks/
      fsmonitor-watchman.sample
      sendemail-validate.sample
      pre-merge-commit.sample
      applypatch-msg.sample
      prepare-commit-msg.sample
      pre-push.sample
      update.sample
      pre-applypatch.sample
      pre-commit.sample
      pre-rebase.sample
      pre-receive.sample
      commit-msg.sample
      push-to-checkout.sample
      post-update.sample
    info/
      exclude
    branches/


In [3]:

%%writefile generate_data.py

import numpy as np
import pandas as pd

# Duka Electronics dataset
rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n, p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n, p=[0.55, 0.35, 0.10]
    )
})

sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

# Mobile money agent dataset
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
    ),
    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d")
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d")
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False), "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("All datasets generated successfully!")


Writing generate_data.py


In [4]:
!python generate_data.py

All datasets generated successfully!


In [5]:

%%writefile cleaning_pipeline.py

import pandas as pd


def clean_agent_data(txns, agents):
    """
    Clean agent transactions and generate a data-quality audit log.
    Returns the cleaned DataFrame and audit log.
    """

    clean_df = txns.copy()
    audit = []

    def add_log(issue, dimension, column, rows_affected):
        audit.append({
            "issue": issue,
            "dimension": dimension,
            "column": column,
            "rows_affected": int(rows_affected)
        })

    # Remove duplicate transaction IDs
    duplicate_count = clean_df["txn_id"].duplicated().sum()
    clean_df = clean_df.drop_duplicates(
        subset="txn_id", keep="first"
    ).copy()

    add_log("Duplicate transaction IDs removed",
            "Uniqueness", "txn_id", duplicate_count)

    # Convert amounts to numeric
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"], errors="coerce"
    )

    # Flag missing amounts
    clean_df["amount_imputed"] = clean_df["amount"].isna()
    missing_count = clean_df["amount_imputed"].sum()

    # Detect outliers using IQR
    q1 = clean_df["amount"].quantile(0.25)
    q3 = clean_df["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    clean_df["is_outlier"] = (
        (clean_df["amount"] < lower_bound) |
        (clean_df["amount"] > upper_bound)
    )

    outlier_count = clean_df["is_outlier"].sum()

    # Impute missing amounts with median
    clean_df["amount"] = clean_df["amount"].fillna(
        clean_df["amount"].median()
    )

    add_log("Missing amounts filled with median",
            "Completeness", "amount", missing_count)

    add_log("Unusual amounts flagged using IQR",
            "Accuracy", "amount", outlier_count)

    # Standardize transaction types
    original_types = clean_df["txn_type"].copy()

    clean_df["txn_type"] = (
        clean_df["txn_type"].astype("string").str.strip().str.title()
    )

    standardized_count = (
        original_types != clean_df["txn_type"]
    ).sum()

    add_log("Transaction types standardized",
            "Consistency", "txn_type", standardized_count)

    # Standardize dates
    original_dates = clean_df["txn_date"].astype("string")

    clean_df["txn_date"] = pd.to_datetime(
        original_dates.str.replace("/", "-", regex=False),
        errors="coerce"
    )

    invalid_dates = clean_df["txn_date"].isna().sum()

    changed_dates = (
        original_dates != clean_df["txn_date"].dt.strftime("%Y-%m-%d")
    ).sum()

    clean_df["txn_date"] = clean_df["txn_date"].dt.strftime("%Y-%m-%d")

    add_log("Transaction dates standardized",
            "Consistency", "txn_date", changed_dates)

    add_log("Invalid transaction dates",
            "Validity", "txn_date", invalid_dates)

    # Merge county information
    clean_df = clean_df.merge(
        agents[["agent_id", "county"]],
        on="agent_id",
        how="left",
        validate="many_to_one"
    )

    unknown_count = clean_df["county"].isna().sum()
    clean_df["county"] = clean_df["county"].fillna("Unknown")

    add_log("Transactions linked to unregistered agents",
            "Validity", "agent_id", unknown_count)

    log = pd.DataFrame(
        audit,
        columns=["issue", "dimension", "column", "rows_affected"]
    )

    return clean_df, log


Writing cleaning_pipeline.py


In [6]:

import pandas as pd
import os
from cleaning_pipeline import clean_agent_data

# Load original datasets
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

# Clean the transactions
clean_df, audit_log = clean_agent_data(txns, agents)

# Save cleaned dataset
os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)

# County summary
county_summary = clean_df.groupby(
    ["county", "txn_type"]
).agg(
    transaction_count=("txn_id", "count"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean")
).reset_index()

# Weekly transaction pivot
clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)

weekly_pivot = clean_df.pivot_table(
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

# Display results
print("CLEANED DATA:", clean_df.shape)
print("\nAUDIT LOG")
display(audit_log)

print("\nCOUNTY SUMMARY")
display(county_summary)

print("\nWEEKLY PIVOT")
display(weekly_pivot)

print("\nSAVED FILE:",
      os.path.exists("outputs/agent_transactions_clean.csv"))


CLEANED DATA: (600, 9)

AUDIT LOG


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs removed,Uniqueness,txn_id,12
1,Missing amounts filled with median,Completeness,amount,20
2,Unusual amounts flagged using IQR,Accuracy,amount,40
3,Transaction types standardized,Consistency,txn_type,376
4,Transaction dates standardized,Consistency,txn_date,5
5,Invalid transaction dates,Validity,txn_date,0
6,Transactions linked to unregistered agents,Validity,agent_id,7



COUNTY SUMMARY


,county,txn_type,transaction_count,total_amount,average_amount
0,Kiambu,Deposit,43,334737.0,7784.581395
1,Kiambu,Withdrawal,27,77050.5,2853.722222
2,Kisumu,Deposit,74,436420.5,5897.574324
3,Kisumu,Withdrawal,37,76374.5,2064.175676
4,Mombasa,Deposit,72,148986.0,2069.250000
5,Mombasa,Withdrawal,61,368592.0,6042.491803
6,Nairobi,Deposit,118,543590.0,4606.694915
7,Nairobi,Withdrawal,62,162036.0,2613.483871
8,Nakuru,Deposit,60,184890.0,3081.500000
9,Nakuru,Withdrawal,39,328594.5,8425.500000



WEEKLY PIVOT


week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0



SAVED FILE: True


# Data Quality Report

## 1. Completeness
The transaction dataset contained 20 missing transaction amounts after duplicate removal. These were replaced using the median amount. The cleaned dataset contains no missing values.

## 2. Uniqueness
The original dataset contained 12 duplicate transaction IDs. Duplicate records were removed by retaining the first occurrence of each transaction ID, leaving 600 unique transactions.

## 3. Consistency
A total of 376 transaction types were standardized to either `Deposit` or `Withdrawal`. Five transaction dates were standardized to the ISO format (`YYYY-MM-DD`).

## 4. Validity
Seven transactions were associated with an unregistered agent ID (`AG099`). These transactions were retained and their county was labelled `Unknown`. No invalid transaction dates were found.

## 5. Accuracy
The IQR method identified 40 unusual transaction amounts. These were flagged for further investigation rather than deleted because an outlier is not necessarily an error.

## 6. Timeliness
The transaction records cover March and April 2026. Dates were standardized to support consistent chronological and weekly analysis.

## Data Privacy and Protection
Customer phone numbers are personal data under Kenya's Data Protection Act, 2019. If collected, they should be protected through encryption, restricted access, data minimization, secure storage and appropriate retention policies. Aggregated or anonymized information should be used for reporting wherever possible.